# Praktikum Pengantar Pembelajaran Mesin

**Nama: Sayyidah Fatimah Azzahra**

**NIM: 235150200111064**

**Kelas: KAL-E**

---
## Bab 6. Support Vector Machine (SVM) Lanjutan


### 1) Import Data

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
url = "https://gist.githubusercontent.com/Thanatoz-1/9e7fdfb8189f0cdf5d73a494e4a6392a/raw/aaecbd14aeaa468cd749528f291aa8a30c2ea09e/iris_dataset.csv"

Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [3]:
data = pd.read_csv(url)



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [4]:
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [5]:
print(data.shape)
print(data.columns)

(150, 5)
Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target'],
      dtype='object')


## 2) Membagi data menjadi data latih dan data uji

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**.


In [6]:
from sklearn.model_selection import train_test_split

data_latih, data_uji = train_test_split(data, test_size=0.2, random_state=101)
data_latih = data_latih.reset_index(drop=True)
data_uji = data_uji.reset_index(drop=True)

Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 120 data, dan **data_uji** terdiri dari 30 data

In [7]:
print(data_latih.shape[0])
print(data_uji.shape[0])

120
30


Pisahkan label/kelas dari data uji menjadi sebuah variabel bernama **label_uji**

In [8]:
fitur_uji = data_uji.drop('target', axis=1)
label_latih = data_latih.pop('target')
label_uji = data_uji.pop('target')

In [9]:
print(data_latih.shape)
print(data_latih.columns)
print(data_uji.shape)

(120, 4)
Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')
(30, 4)


## 3) Pembentukan data latih one-vs-rest

Metode one-vs-rest memerlukan tiga jenis data latih yang diperlukan untuk melatih tiga SVM yang berbeda pada dataset Iris. Fungsi **buat_trainingset** digunakan untuk membentuk tiga dataset tersebut.

In [10]:
def buat_trainingset(dataset, labels):
    trainingset = {}
    list_kelas = labels.unique()
    for kelas in list_kelas:
        data_temp = dataset.copy(deep=True)
        data_temp['target'] = labels.map({kelas: 1}).fillna(-1)
        trainingset[kelas] = data_temp
    return trainingset

Gunakan fungsi **buat_trainingset** untuk membentuk data latih dengan nama variabel **trainingset** yang akan digunakan pada proses training.

In [11]:
trainingset = buat_trainingset(data_latih, label_latih)

Tampilkan isi **trainingset** agar Anda dapat memahami struktur dari variabel tersebut.

In [12]:
print(trainingset)

{'Iris-virginica':      sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                  6.5               3.0                5.8               2.2   
1                  5.5               2.5                4.0               1.3   
2                  6.5               3.0                5.5               1.8   
3                  5.8               2.7                3.9               1.2   
4                  6.8               3.0                5.5               2.1   
..                 ...               ...                ...               ...   
115                6.1               2.9                4.7               1.4   
116                5.9               3.2                4.8               1.8   
117                5.5               2.4                3.7               1.0   
118                4.8               3.4                1.6               0.2   
119                5.7               3.0                4.2               1.2   

     tar

In [13]:
print(trainingset[list(trainingset.keys())[0]].shape)
print(trainingset[list(trainingset.keys())[0]].columns)

(120, 5)
Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target'],
      dtype='object')


## 4) Pembentukan SVM Biner

Tujuan dari algoritma SVM adalah meminimalkan nilai *cost function*. Penghitungan nilai minimal dapat dapat dilakukan dengan menghitung nilai gradien dari *cost function* terlebih dahulu. Fungsi di bawah ini berguna untuk menghitung nilai gradien cost function

In [14]:
def hitung_cost_gradient(W,X,Y, regularization):
    jarak = 1 - (Y*np.dot(X,W))
    dw = np.zeros(len(W))
    if max(0,jarak)==0:
        di=W
    else:
        di = W - (regularization * Y*X)
    dw += di
    return dw

Terdapat beberapa cara untuk meminimalkan nilai *cost function*, salah satunya menggunakan Stochastic Gradient Descent (SGD) untuk melakukan minimasi. Minimasi *cost function* merupakan inti dari algoritma SVM. Fungsi di bawah ini merupakan implementasi algoritma SGD

In [15]:
from sklearn.utils import shuffle

def sgd(data_latih, label_latih, learning_rate=0.000001, max_epoch=1000, regularization=10000):
    data_latih = data_latih.to_numpy()
    label_latih = label_latih.to_numpy()
    bobot = np.zeros(data_latih.shape[1])
    for epoch in range(1, max_epoch):
        X,Y =shuffle(data_latih, label_latih, random_state=101)
        for index, x in enumerate(X):
            delta=hitung_cost_gradient(bobot,x,Y[index], regularization)
            bobot = bobot - (learning_rate * delta)
    return bobot

## 5) Proses Training

Proses training dilakukan dengan memanggil fungsi **sgd** berulang kali sesuai banyaknya kelas yang ada pada data. Dengan demikian, proses training menghasilkan bobot sebanyak kelas yang ada pada dataset. Buatlah fungsi bernama **training** yang digunakan untuk melakukan proses training one-vs-rest

In [16]:
def training(trainingset):
    list_kelas = trainingset.keys()
    w = {}
    for kelas in list_kelas:
        data_latih = trainingset[kelas]
        label_latih = data_latih.pop('target')
        w[kelas] = sgd(data_latih, label_latih)
    return w

Lakukan proses training dengan memanggil fungsi **training** dan menempatkan hasilnya pada variabel **W**

In [17]:
W = training(trainingset)

Tampilkan isi variabel **W**

In [18]:
print(W)

{'Iris-virginica': array([-2.65010023, -4.29029998,  4.01500644,  5.21657315]), 'Iris-versicolor': array([ 0.84397619, -1.99376017,  1.62025302, -4.18078819]), 'Iris-setosa': array([ 0.1928279 ,  0.73145361, -1.08118451, -0.52704199])}


## 6) Proses *testing* biner
Proses testing dilakukan dengan menghitung nilai [*dot product*](https://en.wikipedia.org/wiki/Dot_product) antara bobot hasil training dengan data uji. Kelas data ditentukan berdasarkan tanda (positif atau negatif) dari hasil dot product tersebut. Fungsi berikut mengimplementasikan proses testing

In [19]:
def testing(W,data_uji):
  prediksi = np.array([])
  for i in range(data_uji.shape[0]):
    y_prediksi = np.sign(np.dot(W,data_uji.to_numpy()[i]))
    prediksi = np.append(prediksi,y_prediksi)
  return prediksi

## TUGAS
Pada tugas kali ini Anda mendefinisikan proses testing pada metode one-vs-rest. Proses testing pada metode one-vs-rest dilakukan dengan memanggil proses testing biner untuk setiap **value** pada dictionary **W**. Kelas pada sebuah data latih adalah **key** pada dictionary **W** yang memiliki nilai prediksi **1**. Lengkapi fungsi **testing_onevsrest** di bawah ini. Output dari fungsi tersebut adalah list nama kelas hasil prediksi.

In [20]:
def testing_onevsrest(W, data_uji):
  kelas_prediksi = []
  for i in range(data_uji.shape[0]):
    max_score = -float('inf')
    predicted_class = None
    for key, value in W.items():
      y_prediction = np.sign(np.dot(value, data_uji.to_numpy()[i]))
      if y_prediction == 1:
        score = np.dot(value, data_uji.to_numpy()[i])
        if score > max_score:
          max_score = score
          predicted_class = key
    kelas_prediksi.append(predicted_class)
  return kelas_prediksi

In [21]:
prediksi = testing_onevsrest(W,data_uji)

Berapa banyak data latih yang berhasil diprediksi dengan benar?

In [22]:
prediksi_benar = sum(prediksi == label_uji)

In [23]:
akurasi = sum(prediksi == label_uji) / len(label_uji)
print(f"Jumlah prediksi yang benar: {prediksi_benar}/{len(label_uji)}")
print(akurasi)

Jumlah prediksi yang benar: 26/30
0.8666666666666667


### KESIMPULAN

Pada praktikum kali ini metode SVM telah dipraktikkan untuk melakukan klasifikasi terhadap dateset dengan 3 class. Pada dasarnya SVM hanya mampu mengklasifikasikan data dengan class biner (hanya 2 class), namun dengan strategi One-vs-rest atau One-vs-one, SVM dapat juga digunakan untuk mengklasifikasikan dataset dengan 3 class.

Setelah melakukan semua percobaan ini maka mahasiswa diwajibkan membuat kesimpulan.
Kesimpulan dari percobaan ini merupakan bagian penting untuk merangkum temuan utama
dan relevansinya dengan tujuan percobaan. Mahasiswa harus mampu menyimpulkan:

• Apakah tujuan percobaan telah tercapai? <br>
• Hubungan antara hasil percobaan dan dasar teori. <br>
• Faktor-faktor yang memengaruhi hasil percobaan, termasuk kemungkinan kesalahan yang
terjadi. <br>

Kesimpulan harus disusun secara singkat, jelas, dan berdasarkan data yang diperoleh.

Jawaban:

Saya berhasil menyelesaikan praktikum ini dengan mengimplementasikan SVM untuk mengklasifikasikan dataset Iris menggunakan strategi one-vs-rest, mencapai akurasi 86.67% (26/30 data uji benar). Tujuan untuk memahami dan menerapkan SVM multi-kelas sudah tercapai, sesuai dengan teori yang menyatakan SVM dapat diperluas dari klasifikasi biner ke multi-kelas dengan pendekatan ini. Hasil menunjukkan model mampu menangkap pola data dengan cukup baik. Faktor-faktor yang memengaruhi hasil mungkin di antaranya kurangnya normalisasi fitur dan nilai hyperparameter statis (learning rate dan regularisasi) mungkin menyebabkan 4 data salah prediksi.

O ya, saya mengubah fungsi buat_trainingset dari versi awal yang mengasumsikan kolom kelas ada di dataset menjadi versi yang menerima dataset dan labels terpisah. Perubahan ini membuat fungsi lebih fleksibel dan sesuai dengan alur praktikum saya, di mana label sudah dipisahkan sebelumnya. Hasilnya tetap sama—membentuk trainingset one-vs-rest dengan benar—tetapi dengan cara yang lebih robust dan konsisten dengan proses saya.